# BT4012 CatBoost submission

This notebook runs the final CatBoost pipeline for the BT4012 Bitcoin transaction competition. It checks the data, evaluates four chronological windows, fits the final model, and writes a submission CSV.

Download `train.csv`, `test.csv` and `sample_submission.csv` from Kaggle into `data/` at the repository root. Run the notebook from the repository root after installing `requirements.txt`. The transaction edge list is not needed.

In [ ]:
from pathlib import Path
import json
import os
import statistics

import pandas as pd
from IPython.display import display

from experiments.catboost_pipeline import (
    load_data,
    make_submission,
    new_model,
    run_validation,
)

In [ ]:
DATA_DIR = Path(os.getenv("BT4012_DATA_DIR", "data"))
OUTPUT_DIR = Path(os.getenv("BT4012_OUTPUT_DIR", "outputs"))
OUTPUT_PATH = OUTPUT_DIR / "submission.csv"
MAX_ITERATIONS = int(os.getenv("BT4012_MAX_ITERATIONS", "1500"))
EARLY_STOPPING_ROUNDS = 150

train, test, sample, features = load_data(DATA_DIR)
labelled = train.loc[train["label"].notna()].copy()

summary = pd.DataFrame([
    {"item": "training rows", "count": len(train)},
    {"item": "labelled rows", "count": len(labelled)},
    {"item": "illicit labelled rows", "count": int((labelled["label"] == 1).sum())},
    {"item": "test rows", "count": len(test)},
    {"item": "numerical features", "count": len(features)},
])
display(summary)

## Chronological validation

Each validation window uses only earlier periods for fitting. The windows overlap and were reused during development, so their results are model-selection evidence rather than independent cross-validation estimates.

In [ ]:
validation_results = run_validation(
    train,
    features,
    max_iterations=MAX_ITERATIONS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose=False,
)

display(pd.DataFrame(validation_results))

final_iterations = max(
    1,
    int(statistics.median(item["best_iterations"] for item in validation_results)),
)
print(f"Final iteration count: {final_iterations}")

## Fit and write the submission

The final model is refit on every row with a known label. Predictions are aligned to the sample-submission index and saved as probabilities.

In [ ]:
final_model = new_model(final_iterations, verbose=False)
final_model.fit(labelled[features], labelled["label"].astype("int8"))
probabilities = final_model.predict_proba(test[features])[:, 1]

submission = make_submission(sample, test["index"], probabilities)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

metrics = {
    "model": "CatBoostClassifier",
    "depth": 8,
    "seed": 4012,
    "feature_count": len(features),
    "labelled_training_rows": len(labelled),
    "unlabelled_training_rows_excluded": int(train["label"].isna().sum()),
    "test_rows": len(test),
    "final_iterations": final_iterations,
    "validation": validation_results,
    "submission": str(OUTPUT_PATH),
}
METRICS_PATH = OUTPUT_PATH.with_suffix(".metrics.json")
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

display(submission.head())
print(f"Wrote {len(submission)} predictions to {OUTPUT_PATH}")
print(f"Wrote validation metrics to {METRICS_PATH}")